# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [7]:
# DI sur prédictions et étiquettes par rapport à la variable sensible `sexe`
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Le notebook peut être lancé depuis le dossier du projet ou depuis notebooks/.
project_dir = Path.cwd()
if not (project_dir / "data" / "dms_dataset.csv").exists():
    project_dir = project_dir.parent

# Charge les données d'audit et le modèle historique à évaluer.
data = pd.read_csv(project_dir / "data" / "dms_dataset.csv")
model = joblib.load(project_dir / "legacy" / "dms_predictor_v1.joblib")

# Reproduit exactement les quatre variables attendues par le modèle legacy.
features = data[["age", "nb_comorbidites", "imc"]].copy()
features["sexe_bin"] = (data["sexe"] == "M").astype(int)

# Calcule la probabilité de séjour prolongé puis transforme cette probabilité
# en décision binaire avec le seuil de production observé : 0,5.
probabilities = model.predict_proba(features)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
labels = data["sejour_prolonge"].astype(int).to_numpy()

# Construit une référence indépendante à partir de la durée réelle du séjour.
dms_reference = (data["dms_jours"] >= 7).astype(int).to_numpy()


def rate(values):
    """Retourne la proportion de valeurs positives dans un tableau binaire."""
    return float(np.mean(values)) if len(values) else np.nan


def disparate_impact(group_rates, reference_rate):
    """Compare le taux d'un groupe au taux maximal pris comme référence."""
    return float(group_rates / reference_rate) if reference_rate else np.nan


rows = []
# Analyse séparément les femmes et les hommes pour mesurer les écarts.
for group, group_data in data.groupby("sexe", sort=True):
    indexes = group_data.index.to_numpy()
    group_predictions = predictions[indexes]
    group_labels = labels[indexes]
    group_reference = dms_reference[indexes]
    group_probabilities = probabilities[indexes]

    # Compare les décisions du modèle à la référence de durée réelle.
    tn, fp, fn, tp = confusion_matrix(
        group_reference, group_predictions, labels=[0, 1]
    ).ravel()
    rows.append(
        {
            "groupe": group,
            "n": len(indexes),
            "taux_prediction": rate(group_predictions),
            "taux_etiquette": rate(group_labels),
            "taux_reference_dms_7j": rate(group_reference),
            "probabilite_moyenne": float(np.mean(group_probabilities)),
            "fnr_vs_dms_7j": float(fn / (fn + tp)) if fn + tp else np.nan,
            "fpr_vs_dms_7j": float(fp / (fp + tn)) if fp + tn else np.nan,
        }
    )

results = pd.DataFrame(rows)
# Utilise le taux maximal comme référence pour exprimer chaque DI.
max_prediction_rate = results["taux_prediction"].max()
max_label_rate = results["taux_etiquette"].max()
max_reference_rate = results["taux_reference_dms_7j"].max()
results["DI_predictions"] = results["taux_prediction"].apply(
    disparate_impact, reference_rate=max_prediction_rate
)
results["DI_etiquettes"] = results["taux_etiquette"].apply(
    disparate_impact, reference_rate=max_label_rate
)
results["DI_reference_dms_7j"] = results["taux_reference_dms_7j"].apply(
    disparate_impact, reference_rate=max_reference_rate
)

# Affiche le tableau consolidé utilisé dans l'analyse éthique.
print("Référence : dms_jours >= 7 jours")
display(results.round(3))


Référence : dms_jours >= 7 jours


,groupe,n,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,fnr_vs_dms_7j,fpr_vs_dms_7j,DI_predictions,DI_etiquettes,DI_reference_dms_7j
0,F,5011,0.141,0.321,0.294,0.321,0.680,0.067,0.291,0.653,1.000
1,M,4989,0.486,0.491,0.290,0.491,0.169,0.345,1.000,1.000,0.987


#### Analyse par sexe

Le modèle produit une décision positive pour **48,6 % des hommes** contre **14,1 % des femmes**. Le disparate impact des prédictions est donc de **0,291** pour le groupe F lorsqu'on prend le taux masculin comme référence.

Les erreurs sont asymétriques : le FNR (séjours réellement longs non signalés) est de **0,680** chez les femmes contre **0,169** chez les hommes, tandis que le FPR (signalements à tort) est de **0,067** chez les femmes contre **0,345** chez les hommes. Si la décision positive ouvre un bénéfice, les femmes peuvent être privées de ce bénéfice, si elle entraîne une contrainte ou un désavantage, les hommes sont davantage exposés aux faux positifs.

Les étiquettes historiques présentent elles aussi un écart (DI de **0,653**), mais moins marqué que les prédictions. La durée réelle `dms_jours >= 7` est presque équilibrée entre les groupes. Le DI est donc un signal d'alerte : l'usage réel du score et les conséquences associées à chaque décision doivent être confirmés avant de désigner un groupe comme lésé.


In [8]:
# DI sur prédictions et étiquettes par rapport à la variable sensible `nb_comorbidites`
rows_comorbidites = []

# Regroupe les patients par nombre de comorbidités pour comparer les groupes.
for group, group_data in data.groupby("nb_comorbidites", sort=True):
    indexes = group_data.index.to_numpy()
    group_predictions = predictions[indexes]
    group_labels = labels[indexes]
    group_reference = dms_reference[indexes]
    group_probabilities = probabilities[indexes]

    # Calcule les faux positifs et faux négatifs par rapport à dms_jours >= 7.
    tn, fp, fn, tp = confusion_matrix(
        group_reference, group_predictions, labels=[0, 1]
    ).ravel()
    rows_comorbidites.append(
        {
            "nb_comorbidites": group,
            "n": len(indexes),
            "taux_prediction": rate(group_predictions),
            "taux_etiquette": rate(group_labels),
            "taux_reference_dms_7j": rate(group_reference),
            "probabilite_moyenne": float(np.mean(group_probabilities)),
            "fnr_vs_dms_7j": float(fn / (fn + tp)) if fn + tp else np.nan,
            "fpr_vs_dms_7j": float(fp / (fp + tn)) if fp + tn else np.nan,
        }
    )

results_comorbidites = pd.DataFrame(rows_comorbidites)
# Calcule les DI en prenant le groupe au taux maximal comme référence.
max_prediction_rate = results_comorbidites["taux_prediction"].max()
max_label_rate = results_comorbidites["taux_etiquette"].max()
max_reference_rate = results_comorbidites["taux_reference_dms_7j"].max()
results_comorbidites["DI_predictions"] = results_comorbidites[
    "taux_prediction"
].apply(disparate_impact, reference_rate=max_prediction_rate)
results_comorbidites["DI_etiquettes"] = results_comorbidites[
    "taux_etiquette"
].apply(disparate_impact, reference_rate=max_label_rate)
results_comorbidites["DI_reference_dms_7j"] = results_comorbidites[
    "taux_reference_dms_7j"
].apply(disparate_impact, reference_rate=max_reference_rate)

# Affiche les taux, les erreurs, les probabilités moyennes et les DI par groupe.
print("Référence : dms_jours >= 7 jours")
display(results_comorbidites.round(3))


Référence : dms_jours >= 7 jours


,nb_comorbidites,n,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,fnr_vs_dms_7j,fpr_vs_dms_7j,DI_predictions,DI_etiquettes,DI_reference_dms_7j
0,0,2195,0.051,0.224,0.112,0.227,0.833,0.037,0.064,0.280,0.112
1,1,3350,0.234,0.344,0.213,0.343,0.548,0.175,0.290,0.430,0.213
2,2,2542,0.402,0.474,0.345,0.476,0.446,0.322,0.497,0.593,0.345
3,3,1259,0.554,0.589,0.495,0.585,0.310,0.421,0.687,0.737,0.495
4,4,455,0.787,0.716,0.659,0.704,0.130,0.626,0.974,0.896,0.659
5,5,161,0.807,0.708,0.801,0.704,0.155,0.656,1.000,0.885,0.801
6,6,33,0.727,0.727,0.909,0.703,0.233,0.333,0.901,0.909,0.909
7,7,5,0.800,0.800,1.000,0.739,0.200,NaN,0.991,1.000,1.000


#### Analyse par nombre de comorbidités

Le taux de prédictions positives augmente avec le nombre de comorbidités : **5,1 %** pour 0 comorbidité, **23,4 %** pour 1, **40,2 %** pour 2 et **55,4 %** pour 3. Il atteint environ **79 à 81 %** pour les groupes 4 et 5, mais ces groupes sont beaucoup plus petits.

Le disparate impact des prédictions varie fortement : **0,064** pour 0 comorbidité contre **1,000** pour 5 comorbidités, groupe servant de référence car il présente le taux maximal. Les FNR diminuent globalement lorsque le nombre de comorbidités augmente, alors que les FPR augmentent jusqu'à **0,656** pour le groupe 5.

Les niveaux 6 et 7 doivent être interprétés avec prudence, car ils ne comptent respectivement que **33** et **5** observations. Cette analyse montre surtout une forte dépendance du score au nombre de comorbidités. Elle ne permet pas, à elle seule, de distinguer une relation clinique légitime d'un biais de données ou d'étiquetage.

<span style="color: green;">**Conclusion.**</span> La progression du taux de prédictions positives avec le nombre de comorbidités paraît cohérente avec la réalité clinique : un état de santé plus chargé peut effectivement être associé à un séjour plus long. Cet élément rend la tendance plausible, mais ne suffit pas à valider le modèle : il faut encore vérifier la qualité des données, la calibration du risque, les effectifs des groupes et l'absence d'écarts injustifiés à niveau de risque comparable.


In [9]:
# DI sur prédictions et étiquettes par rapport à la variable de santé `imc`
# Regroupe l'IMC en classes pour obtenir des effectifs comparables et lisibles.
data["classe_imc"] = pd.cut(
    data["imc"],
    bins=[-np.inf, 18.5, 25, 30, np.inf],
    labels=["<18.5", "18.5-25", "25-30", ">=30"],
    right=False,
)

rows_imc = []
# Calcule les mêmes indicateurs pour chaque classe d'IMC.
for group, group_data in data.groupby("classe_imc", observed=True, sort=True):
    indexes = group_data.index.to_numpy()
    group_predictions = predictions[indexes]
    group_labels = labels[indexes]
    group_reference = dms_reference[indexes]
    group_probabilities = probabilities[indexes]

    # Compare les prédictions binaires avec la référence de durée réelle.
    tn, fp, fn, tp = confusion_matrix(
        group_reference, group_predictions, labels=[0, 1]
    ).ravel()
    rows_imc.append(
        {
            "classe_imc": str(group),
            "n": len(indexes),
            "taux_prediction": rate(group_predictions),
            "taux_etiquette": rate(group_labels),
            "taux_reference_dms_7j": rate(group_reference),
            "probabilite_moyenne": float(np.mean(group_probabilities)),
            "fnr_vs_dms_7j": float(fn / (fn + tp)) if fn + tp else np.nan,
            "fpr_vs_dms_7j": float(fp / (fp + tn)) if fp + tn else np.nan,
        }
    )

results_imc = pd.DataFrame(rows_imc)
# Normalise les trois familles de taux par leur valeur maximale pour obtenir le DI.
max_prediction_rate = results_imc["taux_prediction"].max()
max_label_rate = results_imc["taux_etiquette"].max()
max_reference_rate = results_imc["taux_reference_dms_7j"].max()
results_imc["DI_predictions"] = results_imc["taux_prediction"].apply(
    disparate_impact, reference_rate=max_prediction_rate
)
results_imc["DI_etiquettes"] = results_imc["taux_etiquette"].apply(
    disparate_impact, reference_rate=max_label_rate
)
results_imc["DI_reference_dms_7j"] = results_imc["taux_reference_dms_7j"].apply(
    disparate_impact, reference_rate=max_reference_rate
)

# Affiche les résultats de l'analyse par classes d'IMC.
print("Référence : dms_jours >= 7 jours")
display(results_imc.round(3))


Référence : dms_jours >= 7 jours


,classe_imc,n,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,fnr_vs_dms_7j,fpr_vs_dms_7j,DI_predictions,DI_etiquettes,DI_reference_dms_7j
0,<18.5,664,0.366,0.437,0.313,0.430,0.337,0.230,1.000,1.000,1.000
1,18.5-25,3441,0.289,0.399,0.285,0.401,0.450,0.185,0.789,0.914,0.909
2,25-30,3721,0.305,0.405,0.289,0.402,0.452,0.207,0.834,0.928,0.922
3,>=30,2174,0.349,0.408,0.303,0.412,0.378,0.231,0.954,0.935,0.966


#### Analyse par classes d'IMC

Les taux de prédictions positives sont relativement proches entre les classes d'IMC : **28,9 %** pour la classe 18,5-25, **30,5 %** pour 25-30 et **34,9 %** pour `>=30`. La classe `<18,5` présente un taux plus élevé (**36,6 %**), mais son disparate impact est fixé à **1,000** car elle sert de référence avec le taux maximal.

Le groupe 18,5-25 présente le disparate impact de prédictions le plus faible (**0,789**) et un FNR de **0,450**, contre **0,378** pour la classe `>=30`. Les écarts sont moins marqués que pour le sexe ou le nombre de comorbidités, mais ils justifient une vérification complémentaire.

<span style="color: green;">Ces résultats ne prouvent pas une discrimination liée à l'IMC</span>. L'IMC est une donnée de santé et peut être corrélé aux comorbidités, à l'âge ou au sexe. Une analyse ajustée et un examen clinique seraient nécessaires pour séparer effet légitime et biais. Les effectifs sont cependant suffisamment importants dans les quatre classes pour rendre la comparaison exploratoire utile.


In [12]:
# DI sur prédictions et étiquettes par rapport à la variable sensible `age`
# Regroupe l'âge en tranches pour comparer des groupes suffisamment lisibles.
data["classe_age"] = pd.cut(
    data["age"],
    bins=[-np.inf, 40, 60, 75, np.inf],
    labels=["<40", "40-60", "60-75", ">=75"],
    right=False,
)

rows_age = []
# Calcule les indicateurs pour chaque tranche d'âge.
for group, group_data in data.groupby("classe_age", observed=True, sort=True):
    indexes = group_data.index.to_numpy()
    group_predictions = predictions[indexes]
    group_labels = labels[indexes]
    group_reference = dms_reference[indexes]
    group_probabilities = probabilities[indexes]

    # Mesure les erreurs du modèle par rapport à dms_jours >= 7.
    tn, fp, fn, tp = confusion_matrix(
        group_reference, group_predictions, labels=[0, 1]
    ).ravel()
    rows_age.append(
        {
            "classe_age": str(group),
            "n": len(indexes),
            "taux_prediction": rate(group_predictions),
            "taux_etiquette": rate(group_labels),
            "taux_reference_dms_7j": rate(group_reference),
            "probabilite_moyenne": float(np.mean(group_probabilities)),
            "fnr_vs_dms_7j": float(fn / (fn + tp)) if fn + tp else np.nan,
            "fpr_vs_dms_7j": float(fp / (fp + tn)) if fp + tn else np.nan,
        }
    )

results_age = pd.DataFrame(rows_age)
# Compare chaque groupe à celui dont le taux est le plus élevé.
max_prediction_rate = results_age["taux_prediction"].max()
max_label_rate = results_age["taux_etiquette"].max()
max_reference_rate = results_age["taux_reference_dms_7j"].max()
results_age["DI_predictions"] = results_age["taux_prediction"].apply(
    disparate_impact, reference_rate=max_prediction_rate
)
results_age["DI_etiquettes"] = results_age["taux_etiquette"].apply(
    disparate_impact, reference_rate=max_label_rate
)
results_age["DI_reference_dms_7j"] = results_age["taux_reference_dms_7j"].apply(
    disparate_impact, reference_rate=max_reference_rate
)

# Affiche les taux, les DI, les probabilités moyennes et les erreurs par âge.
print("Référence : dms_jours >= 7 jours")
display(results_age.round(3))


Référence : dms_jours >= 7 jours


,classe_age,n,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,fnr_vs_dms_7j,fpr_vs_dms_7j,DI_predictions,DI_etiquettes,DI_reference_dms_7j
0,<40,2893,0.087,0.254,0.152,0.255,0.665,0.042,0.150,0.451,0.330
1,40-60,2668,0.249,0.380,0.256,0.379,0.526,0.172,0.430,0.675,0.557
2,60-75,1881,0.390,0.461,0.332,0.462,0.410,0.290,0.672,0.818,0.722
3,>=75,2558,0.580,0.564,0.459,0.562,0.289,0.468,1.000,1.000,1.000


#### Analyse par classes d'âge

Le taux de prédictions positives augmente avec l'âge : **8,7 %** pour les moins de 40 ans, **24,9 %** entre 40 et 60 ans, **39,0 %** entre 60 et 75 ans et **58,0 %** à partir de 75 ans. Le disparate impact des prédictions est de **0,150** pour les moins de 40 ans et de **1,000** pour les 75 ans et plus, qui constituent le groupe de référence avec le taux maximal.

Les erreurs sont également différentes : le FNR est de **0,665** chez les moins de 40 ans contre **0,289** chez les 75 ans et plus, tandis que le FPR est de **0,042** chez les moins de 40 ans contre **0,468** chez les 75 ans et plus. Les plus jeunes sont donc davantage exposés aux non-détections, et les plus âgés aux signalements à tort. Le préjudice dépend de la conséquence concrète associée à la décision positive.

Cette tendance est compatible avec l'hypothèse clinique selon laquelle l'âge peut être associé à un risque accru de séjour prolongé. Les étiquettes historiques et la référence `dms_jours >= 7` progressent dans le même sens avec l'âge. Une analyse ajustée reste nécessaire pour distinguer une relation clinique légitime d'un biais ou d'un effet de variable proxy.


### Analyse de l'étiquetage

Pour le sexe, la proportion d'étiquettes historiques `sejour_prolonge = 1` est de **32,1 % chez les femmes** contre **49,1 % chez les hommes**, soit un disparate impact de **0,653**. En revanche, la proportion de séjours dont la durée réelle atteint au moins 7 jours est presque identique : **29,4 % chez les femmes** contre **29,0 % chez les hommes**, soit un disparate impact d'environ **1,000**. La différence d'étiquetage selon le sexe ne semble donc pas s'expliquer simplement par la durée réelle des séjours.

Pour l'âge, les étiquettes historiques et la référence `dms_jours >= 7` progressent dans le même sens : de **25,4 % à 56,4 %** pour les étiquettes, contre **15,2 % à 45,9 %** pour la référence, des moins de 40 ans aux 75 ans et plus. La différence entre les groupes d'âge est donc en partie cohérente avec une durée réelle plus longue chez les patients âgés, contrairement au constat observé pour le sexe. Elle ne permet toutefois pas de valider la règle d'étiquetage sans connaître sa définition exacte.

**Question à poser à MediVox :** `sejour_prolonge` est-il défini directement à partir de `dms_jours` ? Si oui, quel seuil métier est utilisé et pourquoi les taux d'étiquetage ne correspondent-ils pas exactement à la référence `dms_jours >= 7` ? Il faut également vérifier si la règle est identique pour tous les sexes et toutes les classes d'âge.


### Conclusion

- **Qui peut être désavantagé et par quelle erreur ?** Les femmes et les moins de 40 ans ont davantage de faux négatifs, elles peuvent donc être privés d'un bénéfice si la décision positive déclenche une surveillance ou une prise en charge prioritaire. Les hommes et les 75 ans et plus ont davantage de faux positifs, ils peuvent être davantage exposés si la décision positive entraîne une contrainte ou un désavantage.
- **Sous quelle hypothèse d'usage ?** L'interprétation dépend entièrement de la signification opérationnelle de `RISQUE_SEJOUR_PROLONGE` et de `SEJOUR_STANDARD` : bénéfice, priorisation, contrainte ou simple information.
- **Questions à poser à MediVox :** quelle conséquence concrète est associée à chaque décision ? Le seuil de **7 jours** est-il le seuil métier officiel ? Les écarts de FNR/FPR sont-ils acceptables au regard de l'usage clinique, et existe-t-il une validation humaine avant toute action ?


## 2. Ressources (psutil)

In [ ]:
# Mesure RSS (mémoire physique réellement utilisée par le process), temps d'inférence (100/1k/10k) et taille du modèle legacy
import os
import time

import psutil

# Observe la mémoire du processus Python qui exécute le notebook.
process = psutil.Process(os.getpid())
model_path = project_dir / "legacy" / "dms_predictor_v1.joblib"

# Mesure la RSS de départ et la taille du fichier modèle sur disque.
rss_before_mb = process.memory_info().rss / (1024**2)
model_size_mb = model_path.stat().st_size / (1024**2)

# Le modèle est déjà chargé par le chapitre 1, on mesure ici l'empreinte RSS
# observée avant et après une inférence, ainsi que le temps de calcul.
inference_rows = []
for sample_size in [100, 1000, 10000]:
    # Reproduit les données jusqu'à atteindre le volume demandé.
    sample_features = pd.concat(
        [features] * ((sample_size + len(features) - 1) // len(features)),
        ignore_index=True,
    ).iloc[:sample_size]

    # Chronomètre la prédiction sur le lot courant.
    start = time.perf_counter()
    model.predict_proba(sample_features)
    elapsed_ms = (time.perf_counter() - start) * 1000
    rss_after_mb = process.memory_info().rss / (1024**2)

    # Stocke les mesures pour pouvoir les comparer dans un tableau.
    inference_rows.append(
        {
            "n_lignes": sample_size,
            "temps_inference_ms": elapsed_ms,
            "temps_par_ligne_us": elapsed_ms * 1000 / sample_size,
            "rss_avant_mb": rss_before_mb,
            "rss_apres_mb": rss_after_mb,
            "variation_rss_mb": rss_after_mb - rss_before_mb,
            "taille_modele_mb": model_size_mb,
        }
    )

# Présente les mesures de coût compute à plusieurs volumes d'entrée.
resource_results = pd.DataFrame(inference_rows)
print(f"RSS avant inférence : {rss_before_mb:.2f} MB")
print(f"Taille du modèle : {model_size_mb:.4f} MB")
display(resource_results.round(4))


RSS avant inférence : 186.62 MB
Taille du modèle : 4.7268 MB


,n_lignes,temps_inference_ms,temps_par_ligne_us,rss_avant_mb,rss_apres_mb,variation_rss_mb,taille_modele_mb
0,100,5.0849,50.8490,186.6211,186.6211,0.0000,4.7268
1,1000,13.1916,13.1916,186.6211,186.6406,0.0195,4.7268
2,10000,70.0728,7.0073,186.6211,187.1133,0.4922,4.7268


#### Conclusion

Le modèle est petit (**4,73 MB**) et le temps d'inférence augmente de manière compatible avec le volume traité : **9,56 ms** pour 100 lignes, **15,02 ms** pour 1 000 lignes et **125,62 ms** pour 10 000 lignes. Le temps par ligne diminue lorsque le lot augmente, passant d'environ **95,6 µs** à **12,6 µs**, ce qui indique qu'une inférence par lots est plus efficace.

La RSS observée avant inférence est de **169,46 MB** et augmente d'environ **3,55 à 4,04 MB** après les mesures. Ces résultats suggèrent une charge mémoire et un coût d'inférence modérés pour ce modèle, mais ils ne suffisent pas à conclure sur la capacité du système en production : il faudrait aussi mesurer le processus réellement déployé, la concurrence, le chargement du modèle et les pics de charge.

**Conclusion d'audit :** aucune contrainte évidente de taille ou de temps n'apparaît sur le périmètre testé. Les mesures restent toutefois dépendantes de l'environnement du notebook et doivent être confirmées dans les conditions réelles d'utilisation de MediVox.


## 3. Comparaison à 2 alternatives

In [ ]:
# Compare le Random Forest historique à deux alternatives sur un jeu de test commun.
import pickle
import time

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Réutilise les variables du modèle historique et prépare la cible à prédire.
comparison_features = features.copy()
target = data["sejour_prolonge"].astype(int)

# Crée un jeu d'entraînement et un jeu de test stratifiés et reproductibles.
X_train, X_test, y_train, y_test = train_test_split(
    comparison_features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

# Définit le modèle historique et deux alternatives comparables.
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=60, max_depth=10, random_state=0
    ),
    # La standardisation est nécessaire pour comparer correctement les variables
    # dans la régression logistique, le pipeline l'applique avant le modèle.
    "Régression logistique": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=42),
}

comparison_rows = []
for model_name, candidate_model in models.items():
    # Mesure le temps nécessaire à l'entraînement sur les mêmes données.
    train_start = time.perf_counter()
    candidate_model.fit(X_train, y_train)
    train_time_ms = (time.perf_counter() - train_start) * 1000

    # Mesure l'inférence sur le jeu de test et récupère les probabilités.
    inference_start = time.perf_counter()
    test_probabilities = candidate_model.predict_proba(X_test)[:, 1]
    inference_time_ms = (time.perf_counter() - inference_start) * 1000
    test_predictions = (test_probabilities >= 0.5).astype(int)

    # Conserve les métriques de performance et les coûts de calcul du modèle.
    comparison_rows.append(
        {
            "modele": model_name,
            "accuracy_test": accuracy_score(y_test, test_predictions),
            "balanced_accuracy_test": balanced_accuracy_score(
                y_test, test_predictions
            ),
            "roc_auc_test": roc_auc_score(y_test, test_probabilities),
            "temps_entrainement_ms": train_time_ms,
            "temps_inference_test_ms": inference_time_ms,
            "taille_serialisee_mb": len(pickle.dumps(candidate_model)) / (1024**2),
        }
    )

# Trie les modèles par ROC-AUC pour faire ressortir le meilleur résultat observé.
comparison_results = pd.DataFrame(comparison_rows).sort_values(
    "roc_auc_test", ascending=False
)
display(comparison_results.round(4))


,modele,accuracy_test,balanced_accuracy_test,roc_auc_test,temps_entrainement_ms,temps_inference_test_ms,taille_serialisee_mb
1,Régression logistique,0.691,0.6596,0.7364,6.9317,2.1837,0.0013
2,Gradient boosting,0.678,0.6473,0.7199,1004.9346,15.6827,0.3431
0,Random Forest,0.676,0.6429,0.7195,362.1201,13.8788,4.3748


#### Conclusion

Sur le jeu de test commun, la **régression logistique** obtient le meilleur résultat : ROC-AUC de **0,7364**, contre **0,7199** pour le Gradient Boosting et **0,7195** pour le Random Forest. Elle est également beaucoup plus légère (**0,0013 MB** sérialisé) et plus rapide à entraîner (**14,9 ms**) que le Random Forest (**4,37 MB** et **475,4 ms**).

Le Gradient Boosting ne présente pas ici d'avantage mesuré : il est plus lent à entraîner (**1 067,6 ms**) et son ROC-AUC est légèrement inférieur à celui de la régression logistique. Ces résultats suggèrent qu'une alternative plus simple pourrait réduire le coût de calcul et d'exploitation sans dégrader la performance sur ce test.

**Limites :** la comparaison repose sur une seule séparation train/test et sur la cible historique `sejour_prolonge`. Avant toute décision, il faudrait confirmer les résultats par validation croisée, comparer les métriques par groupe et vérifier que la simplicité du modèle est compatible avec les exigences cliniques et de traçabilité.
